In [ ]:
# Multi-model benchmark training function (optional)
def train_multiple_models(model_names: List[str], train_params: Dict):
    """Train multiple YOLOv8 models for benchmarking"""

    results_summary = []

    for model_name in model_names:
        try:
            logger.info(f"\n{'='*80}")
            logger.info(f"Training {model_name} model...")
            logger.info(f"{'='*80}")

            # Create MLflow run for each model
            with mlflow.start_run(
                run_name=f"yolov8_{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            ):

                # Update training params for this model
                model_params = train_params.copy()
                model_params["name"] = (
                    f'yolov8_{model_name.split("-")[1]}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
                )

                # Log model-specific parameters
                mlflow.log_params(
                    {
                        "model": model_name,
                        "model_type": model_name.split("-")[1],
                    }
                )

                # Load and train model
                model = YOLO(f"{model_name}.pt")
                results = model.train(**model_params)

                # Log results
                run_dir = Path(model_params["project"]) / model_params["name"]
                best_weights = run_dir / "weights" / "best.pt"

                if best_weights.exists():
                    mlflow.log_artifact(
                        str(best_weights), artifact_path="model_weights"
                    )

                    results_summary.append(
                        {
                            "model": model_name,
                            "run_dir": str(run_dir),
                            "best_weights": str(best_weights),
                            "status": "completed",
                        }
                    )

                logger.info(f"✅ {model_name} training completed")

        except Exception as e:
            logger.error(f"❌ {model_name} training failed: {e}")
            results_summary.append(
                {"model": model_name, "status": "failed", "error": str(e)}
            )

    return results_summary


# To run multi-model benchmarking, uncomment below:
# benchmark_models = ['yolov8n-seg', 'yolov8s-seg', 'yolov8m-seg']
# results = train_multiple_models(benchmark_models, TRAIN_PARAMS)


# Visualization of training metrics
def plot_training_metrics(metrics_df: pd.DataFrame):
    """Plot training metrics from results"""

    if metrics_df is None or len(metrics_df) == 0:
        logger.warning("No metrics to plot")
        return

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("YOLOv8 Training Metrics", fontsize=16, fontweight="bold")

    # Plot 1: Loss
    if "train/loss" in metrics_df.columns:
        axes[0, 0].plot(metrics_df["train/loss"], label="Train Loss", marker="o")
        if "val/loss" in metrics_df.columns:
            axes[0, 0].plot(metrics_df["val/loss"], label="Val Loss", marker="s")
        axes[0, 0].set_xlabel("Epoch")
        axes[0, 0].set_ylabel("Loss")
        axes[0, 0].set_title("Loss over Epochs")
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: mAP
    if "metrics/mAP50(B)" in metrics_df.columns:
        axes[0, 1].plot(metrics_df["metrics/mAP50(B)"], label="mAP50", marker="o")
        if "metrics/mAP50-95(B)" in metrics_df.columns:
            axes[0, 1].plot(
                metrics_df["metrics/mAP50-95(B)"], label="mAP50-95", marker="s"
            )
        axes[0, 1].set_xlabel("Epoch")
        axes[0, 1].set_ylabel("mAP Score")
        axes[0, 1].set_title("mAP over Epochs")
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: Precision/Recall
    if "metrics/precision(B)" in metrics_df.columns:
        axes[1, 0].plot(
            metrics_df["metrics/precision(B)"], label="Precision", marker="o"
        )
        if "metrics/recall(B)" in metrics_df.columns:
            axes[1, 0].plot(metrics_df["metrics/recall(B)"], label="Recall", marker="s")
        axes[1, 0].set_xlabel("Epoch")
        axes[1, 0].set_ylabel("Score")
        axes[1, 0].set_title("Precision & Recall over Epochs")
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

    # Plot 4: Segmentation Loss
    if "train/seg_loss" in metrics_df.columns:
        axes[1, 1].plot(
            metrics_df["train/seg_loss"], label="Train Seg Loss", marker="o"
        )
        if "val/seg_loss" in metrics_df.columns:
            axes[1, 1].plot(
                metrics_df["val/seg_loss"], label="Val Seg Loss", marker="s"
            )
        axes[1, 1].set_xlabel("Epoch")
        axes[1, 1].set_ylabel("Segmentation Loss")
        axes[1, 1].set_title("Segmentation Loss over Epochs")
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(run_dir / "training_metrics_plot.png", dpi=300, bbox_inches="tight")
    logger.info(f"✅ Metrics plot saved to {run_dir / 'training_metrics_plot.png'}")
    plt.show()


# Plot metrics if available
if metrics_df is not None and len(metrics_df) > 0:
    plot_training_metrics(metrics_df)

print("\n" + "=" * 80)
print("🎉 YOLOv8 TRAINING PIPELINE COMPLETE!")
print("=" * 80)
print(f"\n📊 MLflow Dashboard:")
print(f"   To view results: open {mlflow.get_tracking_uri()}")
print(f"   Experiment: {EXPERIMENT_NAME}")
print(f"\n💾 Model Artifacts:")
print(f"   Best weights: {best_model_path}")
print(
    f"   Config: {benchmark_config_path if 'benchmark_config_path' in locals() else 'N/A'}"
)
print(f"\n🚀 Next Steps:")
print(f"   1. Review metrics in MLflow UI")
print(f"   2. Evaluate other model sizes (yolov8s-seg, yolov8m-seg)")
print(f"   3. Fine-tune hyperparameters based on results")

## Bonus: Multi-Model Benchmarking and Visualization

Train and compare multiple YOLOv8 model sizes for comprehensive benchmarking.

In [ ]:
class BenchmarkManager:
    """Manage and compare benchmark models"""

    def __init__(self, experiment_name: str):
        self.experiment_name = experiment_name
        self.benchmarks = []

    def create_benchmark(self, model_path: Path, metadata: Dict) -> Dict:
        """Create a benchmark entry for a trained model"""
        try:
            benchmark = {
                "timestamp": datetime.now().isoformat(),
                "model_path": str(model_path),
                "metadata": metadata,
                "model_size_mb": model_path.stat().st_size / (1024 * 1024),
            }

            self.benchmarks.append(benchmark)
            logger.info(f"✅ Benchmark created for {model_path.name}")

            return benchmark

        except Exception as e:
            logger.error(f"Error creating benchmark: {e}")
            return {}

    def compare_benchmarks(self) -> pd.DataFrame:
        """Compare all benchmarks"""
        if not self.benchmarks:
            logger.warning("No benchmarks to compare")
            return pd.DataFrame()

        try:
            comparison_data = []
            for bench in self.benchmarks:
                comparison_data.append(
                    {
                        "timestamp": bench["timestamp"],
                        "model_size_mb": bench["model_size_mb"],
                        "accuracy": bench["metadata"].get("accuracy", None),
                        "mAP50": bench["metadata"].get("mAP50", None),
                        "mAP50_95": bench["metadata"].get("mAP50_95", None),
                    }
                )

            df = pd.DataFrame(comparison_data)
            logger.info(f"✅ Created comparison for {len(self.benchmarks)} benchmarks")

            return df

        except Exception as e:
            logger.error(f"Error comparing benchmarks: {e}")
            return pd.DataFrame()


# Initialize benchmark manager
benchmark_manager = BenchmarkManager(EXPERIMENT_NAME)

# Register benchmark model
if best_model_path.exists():
    benchmark_metadata = {
        "model_name": SELECTED_MODEL,
        "epochs_trained": TRAIN_PARAMS["epochs"],
        "image_size": TRAIN_PARAMS["imgsz"],
        "batch_size": TRAIN_PARAMS["batch"],
        "timestamp": datetime.now().isoformat(),
    }

    # Add evaluation metrics if available
    if eval_metrics:
        benchmark_metadata.update(
            {
                "mAP50": eval_metrics.get("val_map50"),
                "mAP50_95": eval_metrics.get("val_map50_95"),
                "precision": eval_metrics.get("val_precision"),
                "recall": eval_metrics.get("val_recall"),
            }
        )

    benchmark = benchmark_manager.create_benchmark(best_model_path, benchmark_metadata)

    # Save benchmark config
    benchmark_config_path = run_dir / "benchmark_config.json"
    with open(benchmark_config_path, "w") as f:
        json.dump(benchmark_metadata, f, indent=4)

    logger.info(f"✅ Benchmark config saved to {benchmark_config_path}")

    # Log to MLflow
    with mlflow.start_run() as run:
        if run is not None:
            mlflow.log_dict(benchmark_metadata, "benchmark_metadata.json")
            mlflow.log_artifact(str(best_model_path), artifact_path="best_model")

print("\n🏆 Model Training & Benchmarking Summary:")
print(f"   Model: {SELECTED_MODEL}")
print(f"   Training Run: {run_name}")
print(f"   Results Directory: {run_dir}")
print(f"   Best Model: {best_model_path}")
print(f"   MLflow Experiment: {EXPERIMENT_NAME}")
print(f"   MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print("\n✅ Training pipeline completed successfully!")

## Section 9: Save and Register Benchmark Model

Save the trained model as a benchmark, register with MLflow Model Registry, and document metadata.

In [ ]:
class ModelEvaluator:
    """Evaluate trained model on validation and test sets"""

    def __init__(self, model_path: Path, data_yaml: Path):
        self.model_path = Path(model_path)
        self.data_yaml = Path(data_yaml)
        self.model = None
        self.eval_results = {}

    def load_model(self) -> bool:
        """Load trained model"""
        try:
            if not self.model_path.exists():
                logger.error(f"Model not found: {self.model_path}")
                return False

            self.model = YOLO(str(self.model_path))
            logger.info(f"✅ Model loaded: {self.model_path}")
            return True

        except Exception as e:
            logger.error(f"Error loading model: {e}")
            return False

    def evaluate(self) -> Dict:
        """Run evaluation on validation and test sets"""
        if self.model is None:
            logger.error("Model not loaded")
            return {}

        try:
            logger.info("🔍 Running model evaluation...")

            # Validate
            val_results = self.model.val(data=str(self.data_yaml))

            self.eval_results = {
                "val_map50": val_results.results_dict.get("metrics/mAP50(B)", None),
                "val_map50_95": val_results.results_dict.get(
                    "metrics/mAP50-95(B)", None
                ),
                "val_precision": val_results.results_dict.get(
                    "metrics/precision(B)", None
                ),
                "val_recall": val_results.results_dict.get("metrics/recall(B)", None),
            }

            logger.info("✅ Evaluation completed")
            return self.eval_results

        except Exception as e:
            logger.error(f"Evaluation error: {e}\n{traceback.format_exc()}")
            return {}

    def log_evaluation_to_mlflow(self):
        """Log evaluation metrics to MLflow"""
        try:
            for metric_name, value in self.eval_results.items():
                if value is not None:
                    mlflow.log_metric(f"evaluation/{metric_name}", float(value))

            logger.info("✅ Evaluation metrics logged to MLflow")

        except Exception as e:
            logger.error(f"Error logging evaluation metrics: {e}")


# Evaluate model
best_model_path = run_dir / "weights" / "best.pt"

if best_model_path.exists():
    logger.info(f"Evaluating best model: {best_model_path}")
    evaluator = ModelEvaluator(
        best_model_path, results_csv.parent.parent / "dataset_yolo" / "data.yaml"
    )

    if evaluator.load_model():
        eval_metrics = evaluator.evaluate()

        print("\n📈 Evaluation Results:")
        for metric, value in eval_metrics.items():
            if value is not None:
                print(f"   {metric}: {value:.4f}")

        # Log to MLflow
        with mlflow.start_run() as run:
            if run is not None:
                evaluator.log_evaluation_to_mlflow()
else:
    logger.warning(f"Best model not found: {best_model_path}")

## Section 8: Evaluate Model Performance

Run validation and testing on holdout datasets, calculate performance metrics, and log to MLflow.

In [ ]:
class MetricsExtractor:
    """Extract and parse training metrics from YOLOv8 results"""

    def __init__(self, results_csv_path: Path):
        self.results_path = Path(results_csv_path)
        self.metrics_df = None

    def load_and_parse(self) -> Optional[pd.DataFrame]:
        """Load and parse results CSV"""
        try:
            if not self.results_path.exists():
                logger.warning(f"Results file not found: {self.results_path}")
                return None

            self.metrics_df = pd.read_csv(self.results_path)

            # Clean column names (remove extra spaces)
            self.metrics_df.columns = self.metrics_df.columns.str.strip()

            logger.info(f"✅ Loaded results from {self.results_path}")
            logger.info(f"   Shape: {self.metrics_df.shape}")
            logger.info(f"   Columns: {list(self.metrics_df.columns)}")

            return self.metrics_df

        except Exception as e:
            logger.error(f"Error loading results: {e}")
            return None

    def log_to_mlflow(self):
        """Log per-epoch metrics to MLflow"""
        if self.metrics_df is None:
            logger.warning("No metrics to log")
            return

        try:
            mlflow.start_nested_run()

            for idx, row in self.metrics_df.iterrows():
                epoch = int(idx)

                # Log common metrics
                metric_mapping = {
                    "train/box_loss": "train/box_loss",
                    "train/cls_loss": "train/cls_loss",
                    "train/dfl_loss": "train/dfl_loss",
                    "train/seg_loss": "train/seg_loss",
                    "val/box_loss": "val/box_loss",
                    "val/cls_loss": "val/cls_loss",
                    "val/dfl_loss": "val/dfl_loss",
                    "val/seg_loss": "val/seg_loss",
                    "metrics/precision(B)": "metrics/precision",
                    "metrics/recall(B)": "metrics/recall",
                    "metrics/mAP50(B)": "metrics/mAP50",
                    "metrics/mAP50-95(B)": "metrics/mAP50_95",
                }

                for col, metric_name in metric_mapping.items():
                    if col in self.metrics_df.columns:
                        value = row[col]
                        if pd.notna(value):
                            try:
                                mlflow.log_metric(
                                    f"epoch/{metric_name}", float(value), step=epoch
                                )
                            except Exception as e:
                                logger.debug(f"Could not log {metric_name}: {e}")

            logger.info(f"✅ Logged {len(self.metrics_df)} epochs to MLflow")
            mlflow.end_nested_run()

        except Exception as e:
            logger.error(f"Error logging metrics: {e}")


# Find and parse results
run_dir = Path(TRAIN_PARAMS["project"]) / TRAIN_PARAMS["name"]
results_csv = run_dir / "results.csv"

if results_csv.exists():
    logger.info(f"Parsing results from: {results_csv}")
    extractor = MetricsExtractor(results_csv)
    metrics_df = extractor.load_and_parse()

    if metrics_df is not None:
        print("\n📊 Per-Epoch Metrics (First 5 epochs):")
        print(metrics_df.head())

        with mlflow.start_run() as active_run:
            if active_run is None:
                mlflow.start_run(run_name=run_name)
            extractor.log_to_mlflow()
else:
    logger.warning(f"Results file not found: {results_csv}")

## Section 7: Extract and Log Per-Epoch Metrics

Parse training results and log loss, precision, recall, and mAP metrics for each epoch.

In [8]:
# Start MLflow run
run_name = (
    f"yolov8_{SELECTED_MODEL.split('-')[1]}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
)

with mlflow.start_run(run_name=run_name) as run:
    logger.info(f"Started MLflow run: {run.info.run_id}")

    try:
        # Log parameters to MLflow
        mlflow.log_params(
            {
                "model": SELECTED_MODEL,
                "epochs": TRAIN_PARAMS["epochs"],
                "batch_size": TRAIN_PARAMS["batch"],
                "image_size": TRAIN_PARAMS["imgsz"],
                "optimizer": TRAIN_PARAMS["optimizer"],
                "learning_rate": TRAIN_PARAMS["lr0"],
                "device": str(TRAIN_PARAMS["device"]),
            }
        )

        logger.info("🚀 Starting YOLOv8 model training...")
        print("=" * 80)
        print("TRAINING STARTED")
        print("=" * 80)

        # Train model
        results = model.train(**TRAIN_PARAMS)

        logger.info("✅ Training completed successfully!")

        # Log final metrics
        mlflow.log_metric("training_completed", 1)

        print("=" * 80)
        print("TRAINING COMPLETED")
        print("=" * 80)

    except KeyboardInterrupt:
        logger.info("⚠️ Training interrupted by user")
        mlflow.log_metric("training_interrupted", 1)

    except Exception as e:
        logger.error(f"❌ Training failed with error: {e}")
        logger.error(f"Traceback: {traceback.format_exc()}")
        mlflow.log_metric("training_failed", 1)
        raise

    finally:
        # Log artifacts
        logger.info("Logging artifacts to MLflow...")
        run_dir = Path(TRAIN_PARAMS["project"]) / TRAIN_PARAMS["name"]

        if run_dir.exists():
            # Log training results
            results_file = run_dir / "results.csv"
            if results_file.exists():
                mlflow.log_artifact(str(results_file), artifact_path="training_results")

            # Log weights
            weights_dir = run_dir / "weights"
            if weights_dir.exists():
                mlflow.log_artifact(
                    str(weights_dir / "best.pt"), artifact_path="model_weights"
                )
                mlflow.log_artifact(
                    str(weights_dir / "last.pt"), artifact_path="model_weights"
                )

        logger.info(f"✅ MLflow run completed: {run.info.run_id}")

TRAINING STARTED


ERROR:__main__:❌ Training failed with error: 'seg' is not a valid YOLO argument. 

    Arguments received: ['yolo', '--f=c:\\Users\\amans\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3886cfd145265df005c5a3cc60e1a00451dd21f73.json']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of ['detect', 'classify', 'segment', 'obb', 'pose']
                MODE (required) is one of ['val', 'track', 'benchmark', 'predict', 'export', 'train']
                ARGS (optional) are any number of custom 'arg=value' pairs like 'imgsz=320' that override defaults.
                    See all ARGS at https://docs.ultralytics.com/usage/cfg or with 'yolo cfg'

    1. Train a detection model for 10 epochs with an initial learning_rate of 0.01
        yolo train data=coco8.yaml model=yolo26n.pt epochs=10 lr0=0.01

    2. Predict a YouTube video using a pretrained segmentation model at image size 320:
        yolo predict model=yolo26n-s

SyntaxError: '[31m[1mseg[0m' is not a valid YOLO argument. 

    Arguments received: ['yolo', '--f=c:\\Users\\amans\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3886cfd145265df005c5a3cc60e1a00451dd21f73.json']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of ['detect', 'classify', 'segment', 'obb', 'pose']
                MODE (required) is one of ['val', 'track', 'benchmark', 'predict', 'export', 'train']
                ARGS (optional) are any number of custom 'arg=value' pairs like 'imgsz=320' that override defaults.
                    See all ARGS at https://docs.ultralytics.com/usage/cfg or with 'yolo cfg'

    1. Train a detection model for 10 epochs with an initial learning_rate of 0.01
        yolo train data=coco8.yaml model=yolo26n.pt epochs=10 lr0=0.01

    2. Predict a YouTube video using a pretrained segmentation model at image size 320:
        yolo predict model=yolo26n-seg.pt source='https

## Section 6: Train Model with MLflow Logging

Execute model training with MLflow context active, capturing all training logs and metrics.

In [7]:
class MLflowCallback:
    """Custom callback to log YOLOv8 metrics to MLflow at each epoch"""

    def __init__(self):
        self.epoch_metrics = []

    def on_train_epoch_end(self, trainer):
        """Called at the end of each training epoch"""
        try:
            epoch = trainer.epoch
            metrics = trainer.metrics

            # Extract key metrics
            epoch_data = {
                "epoch": epoch,
                "train_loss": metrics.get("train/loss", None),
                "train_box_loss": metrics.get("train/box_loss", None),
                "train_seg_loss": metrics.get("train/seg_loss", None),
                "train_cls_loss": metrics.get("train/cls_loss", None),
                "train_dfl_loss": metrics.get("train/dfl_loss", None),
            }

            # Log to MLflow
            if epoch % 5 == 0 or epoch == 0:  # Log every 5 epochs
                for metric_name, value in epoch_data.items():
                    if value is not None:
                        mlflow.log_metric(metric_name, value, step=epoch)

            self.epoch_metrics.append(epoch_data)

        except Exception as e:
            logger.warning(f"Error in MLflow callback: {e}")


# Training hyperparameters
TRAIN_PARAMS = {
    "data": str(dataset_root / "dataset_yolo" / "data.yaml"),
    "epochs": 60,
    "imgsz": 512,
    "batch": 4,
    "patience": 10,  # Early stopping
    "device": 0 if cuda.is_available() else "cpu",
    "workers": 0 if cuda.is_available() else 0,
    "optimizer": "SGD",  # SGD or Adam
    "lr0": 0.01,  # Initial learning rate
    "lrf": 0.01,  # Final learning rate
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "box": 7.5,  # Box loss weight
    "cls": 0.5,  # Class loss weight
    "dfl": 1.5,  # DFL loss weight
    "seg": 1.0,  # Segmentation loss weight
    "project": str(dataset_root / "runs"),
    "name": f'yolov8_seg_{datetime.now().strftime("%Y%m%d_%H%M%S")}',
    "exist_ok": False,
    "pretrained": True,
    "cache": False,
    "amp": True,  # Automatic Mixed Precision
    "save": True,
    "save_period": 10,
    "verbose": True,
    "seed": 42,
    "deterministic": True,
    "single_cls": False,
}

print("🔧 Training Hyperparameters:")
for key, value in TRAIN_PARAMS.items():
    if key not in ["data", "project", "name"]:  # Don't print file paths
        print(f"   {key}: {value}")

🔧 Training Hyperparameters:
   epochs: 60
   imgsz: 512
   batch: 4
   patience: 10
   device: 0
   workers: 0
   optimizer: SGD
   lr0: 0.01
   lrf: 0.01
   momentum: 0.937
   weight_decay: 0.0005
   warmup_epochs: 3.0
   warmup_momentum: 0.8
   box: 7.5
   cls: 0.5
   dfl: 1.5
   seg: 1.0
   exist_ok: False
   pretrained: True
   cache: False
   amp: True
   save: True
   save_period: 10
   verbose: True
   seed: 42
   deterministic: True
   single_cls: False


## Section 5: Set Up Training Parameters and Custom Callbacks

Define hyperparameters and create custom callbacks to log per-epoch metrics to MLflow.

In [5]:
# Check GPU Availability
print("\n🖥️  GPU/Device Information:")
print(f"   PyTorch version: {torch.__version__}")
print(f"   CUDA available: {cuda.is_available()}")
if cuda.is_available():
    print(f"   CUDA version: {torch.version.cuda}")
    print(f"   GPU count: {cuda.device_count()}")
    for i in range(cuda.device_count()):
        print(f"   GPU {i}: {cuda.get_device_name(i)}")
    print(f"   Current GPU: {cuda.get_device_name(cuda.current_device())}")
else:
    print("   ⚠️  CUDA not available. Training will use CPU (may be slow).")

# Model configuration
MODEL_SIZES = ["yolov8n-seg", "yolov8s-seg", "yolov8m-seg"]  # nano, small, medium
SELECTED_MODEL = "yolov8n-seg"  # Start with nano model

print(f"\n🤖 Available Models for Benchmarking: {MODEL_SIZES}")
print(f"   Selected for training: {SELECTED_MODEL}")

# Load pretrained model
try:
    logger.info(f"Loading {SELECTED_MODEL} model...")
    model = YOLO(f"{SELECTED_MODEL}.pt")
    logger.info(f"✅ Model loaded successfully: {model}")

    # Print model info
    model_info = model.info()
    print(f"\n📋 Model Summary:")
    print(f"   Model: {model.model_name}")
    print(f"   Task: {model.task}")
except Exception as e:
    logger.error(f"Failed to load model: {e}\n{traceback.format_exc()}")
    raise


🖥️  GPU/Device Information:
   PyTorch version: 2.11.0+cu128
   CUDA available: True
   CUDA version: 12.8
   GPU count: 1
   GPU 0: NVIDIA GeForce GTX 1650
   Current GPU: NVIDIA GeForce GTX 1650

🤖 Available Models for Benchmarking: ['yolov8n-seg', 'yolov8s-seg', 'yolov8m-seg']
   Selected for training: yolov8n-seg
YOLOv8n-seg summary: 151 layers, 3,409,968 parameters, 0 gradients, 12.1 GFLOPs

📋 Model Summary:
   Model: yolov8n-seg.pt
   Task: segment


## Section 4: Load and Configure YOLOv8 Segmentation Model

Check GPU availability and load the pretrained YOLOv8 segmentation model.

In [4]:
# MLflow Configuration
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "lettuce_disease_segmentation"
ARTIFACT_PATH = "Leaf Disease Segmentation.v1i.coco-segmentation/Notebooks"

# Set MLflow tracking URI (local file system)
mlflow.set_tracking_uri(f"file:///{notebook_dir / 'mlruns'}")
logger.info(f"✅ MLflow tracking URI set to: {mlflow.get_tracking_uri()}")

# Create or get experiment
try:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        exp_id = mlflow.create_experiment(EXPERIMENT_NAME)
        logger.info(
            f"✅ Created new MLflow experiment: {EXPERIMENT_NAME} (ID: {exp_id})"
        )
    else:
        exp_id = experiment.experiment_id
        logger.info(
            f"✅ Using existing MLflow experiment: {EXPERIMENT_NAME} (ID: {exp_id})"
        )

    mlflow.set_experiment(EXPERIMENT_NAME)
except Exception as e:
    logger.error(f"MLflow initialization error: {e}")
    raise

print(f"\n📊 MLflow Experiment Ready:")
print(f"   Experiment: {EXPERIMENT_NAME}")
print(f"   Tracking URI: {mlflow.get_tracking_uri()}")

d:\gemma4\gemma4\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)



📊 MLflow Experiment Ready:
   Experiment: lettuce_disease_segmentation
   Tracking URI: file:///d:\gemma4\segmentation_lattuce-desease\mlruns


## Section 3: Initialize MLflow Experiment Tracking

Set up MLflow tracking server, create an experiment for the segmentation model, and configure logging.

In [3]:
class DatasetValidator:
    """Validate dataset structure and integrity before training"""

    def __init__(self, dataset_root: Path, data_yaml_path: Path):
        self.dataset_root = Path(dataset_root)
        self.data_yaml_path = Path(data_yaml_path)
        self.validation_report = {}

    def validate(self) -> bool:
        """Run all validation checks"""
        try:
            logger.info("Starting dataset validation...")

            # Check YAML file
            if not self._validate_yaml():
                return False

            # Load YAML and validate structure
            with open(self.data_yaml_path, "r") as f:
                yaml_content = yaml.safe_load(f)

            # Check required keys
            required_keys = ["path", "train", "val", "names"]
            for key in required_keys:
                if key not in yaml_content:
                    logger.error(f"Missing required key '{key}' in data.yaml")
                    return False

            # Check dataset directories
            if not self._validate_directories(yaml_content):
                return False

            # Check for data files
            if not self._validate_data_files(yaml_content):
                return False

            logger.info("✅ Dataset validation PASSED!")
            return True

        except Exception as e:
            logger.error(f"Dataset validation error: {e}\n{traceback.format_exc()}")
            return False

    def _validate_yaml(self) -> bool:
        """Check if data.yaml exists and is readable"""
        if not self.data_yaml_path.exists():
            logger.error(f"data.yaml not found at {self.data_yaml_path}")
            return False

        try:
            with open(self.data_yaml_path, "r") as f:
                yaml.safe_load(f)
            logger.info(f"✅ data.yaml found and valid at {self.data_yaml_path}")
            return True
        except Exception as e:
            logger.error(f"data.yaml is not valid YAML: {e}")
            return False

    def _validate_directories(self, yaml_content: dict) -> bool:
        """Validate train/val/test directories exist"""
        base_path = self.dataset_root / yaml_content["path"]

        for split in ["train", "val"]:
            split_path = base_path / yaml_content[split]
            if not split_path.exists():
                logger.error(f"{split} directory not found: {split_path}")
                return False
            logger.info(f"✅ {split} directory exists: {split_path}")

        return True

    def _validate_data_files(self, yaml_content: dict) -> bool:
        """Validate image and label files exist"""
        base_path = self.dataset_root / yaml_content["path"]

        for split in ["train", "val"]:
            split_path = base_path / yaml_content[split]
            images_dir = split_path / "images"

            if not images_dir.exists():
                logger.error(f"Images directory not found: {images_dir}")
                return False

            image_files = list(images_dir.glob("**/*.jpg")) + list(
                images_dir.glob("**/*.png")
            )
            logger.info(f"✅ {split} split: {len(image_files)} images found")

            if len(image_files) == 0:
                logger.error(f"No images found in {images_dir}")
                return False

        # Log class info
        logger.info(f"Classes: {yaml_content['names']}")
        return True


# Initialize validator
notebook_dir = Path.cwd()
dataset_root = (
    notebook_dir / "Leaf Disease Segmentation.v1i.coco-segmentation" / "Notebooks"
)
data_yaml_path = dataset_root / "dataset_yolo" / "data.yaml"

validator = DatasetValidator(dataset_root, data_yaml_path)
is_valid = validator.validate()

if not is_valid:
    logger.error("⚠️ Dataset validation failed. Check paths and dataset structure.")
    print("Current working directory:", notebook_dir)
    print("Expected dataset_root:", dataset_root)
    print("Actual directories:", list(notebook_dir.glob("**/dataset_yolo*")))

ERROR:__main__:train directory not found: d:\gemma4\segmentation_lattuce-desease\Leaf Disease Segmentation.v1i.coco-segmentation\Notebooks\dataset_yolo\images\train
ERROR:__main__:⚠️ Dataset validation failed. Check paths and dataset structure.


Current working directory: d:\gemma4\segmentation_lattuce-desease
Expected dataset_root: d:\gemma4\segmentation_lattuce-desease\Leaf Disease Segmentation.v1i.coco-segmentation\Notebooks
Actual directories: [WindowsPath('d:/gemma4/segmentation_lattuce-desease/Leaf Disease Segmentation.v1i.coco-segmentation/Notebooks/dataset_yolo')]


## Section 2: Verify Dataset Structure and YAML Configuration

Validate the dataset directory structure, verify data.yaml exists with correct paths and class definitions.

In [2]:
import os
import sys
import json
import yaml
import logging
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from datetime import datetime
import traceback

# ML/Data Libraries
import torch
import torch.cuda as cuda
import numpy as np
import pandas as pd
from PIL import Image

# YOLOv8
from ultralytics import YOLO

from ultralytics.data.converter import convert_coco

# MLflow
import mlflow
import mlflow.pytorch
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import ColSpec, Schema

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# Set up logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)
print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Section 1: Import Required Libraries

Import all necessary libraries for YOLOv8 training, MLflow experiment tracking, data validation, and monitoring.

# YOLOv8 Segmentation Model Training with MLflow

This notebook provides a comprehensive, production-ready training pipeline for YOLOv8 segmentation models with:
- MLflow experiment tracking and metrics logging
- Per-epoch loss and performance metrics capture
- Multi-model benchmarking (nano, small, medium sizes)
- Robust error handling and data validation
- Custom callbacks for detailed monitoring
- Model registry and versioning support

**Dataset**: Lettuce Disease Segmentation (HEALTHY vs LEAF_DISEASE)
**Task**: Instance segmentation with automatic metric logging